In [35]:
import pandas as pd
import psycopg2
from sqlalchemy import create_engine
from datetime import datetime

In [36]:
# Connect PostgreSQL Databse

# Create PostgreSQL Connection
# Add your PostgreSQL credentials below 

from sqlalchemy import URL, create_engine

connection_url = URL.create(
    drivername="postgresql+psycopg2",
    username="postgres",
    password="Spoorti@26",
    host="localhost",
    port=5432,
    database="EnterpriseBankingDataPlatform"
)

engine = create_engine(connection_url)

In [37]:
# Verify

query = """
SELECT CURRENT_TIMESTAMP;
"""

pd.read_sql(query, engine)

,current_timestamp
0,2026-08-05 03:56:21.001072+00:00


# ========================
# Banking Dashboard KPIs
# ========================

In [38]:
# Banking KPIs

banking_kpi_query = """
SELECT
    COUNT(*) AS transaction_count,
    SUM(amount) AS total_transaction_amount,
    COUNT(DISTINCT customer_id) AS customer_count,
    AVG(amount) AS average_transaction_amount
FROM reporting.vw_transactions;
"""

banking_kpis = pd.read_sql(banking_kpi_query, engine)

banking_kpis

,transaction_count,total_transaction_amount,customer_count,average_transaction_amount
0,999991,4.998230e+08,59999,499.82748


In [39]:
# Merchant Category Summary

merchant_query = """
SELECT
    merchant_category,
    COUNT(*) AS transaction_count
FROM reporting.vw_transactions
GROUP BY merchant_category
ORDER BY transaction_count DESC;
"""

merchant_summary = pd.read_sql(merchant_query, engine)

merchant_summary

,merchant_category,transaction_count
0,electronics,200398
1,fashion,200117
2,gaming,199935
3,groceries,199786
4,travel,199755


In [40]:
# Peak Banking Month

peak_month_query = """
SELECT
    month_name,
    COUNT(*) AS transaction_count
FROM reporting.vw_transactions
GROUP BY month_name, month
ORDER BY transaction_count DESC
LIMIT 1;
"""

peak_month = pd.read_sql(peak_month_query, engine)

peak_month

,month_name,transaction_count
0,May,85324


In [41]:
# Fraud KPIs

fraud_kpi_query = """
SELECT
    COUNT(*) FILTER (WHERE "isFraud" = 1) AS fraud_transaction_count,
    SUM(amount) FILTER (WHERE "isFraud" = 1) AS fraud_amount,
    AVG(fraud_probability) FILTER (WHERE "isFraud" = 1) AS average_fraud_probability,
    COUNT(*) FILTER (WHERE "isMoneyLaundering" = 1) AS money_laundering_cases
FROM reporting.vw_aml_transactions;
"""

fraud_kpis = pd.read_sql(fraud_kpi_query, engine)

fraud_kpis

,fraud_transaction_count,fraud_amount,average_fraud_probability,money_laundering_cases
0,1411,1.420383e+07,0.726119,1411


In [42]:
# Fraud Typology Summary

typology_query = """
SELECT
    typology,
    COUNT(*) AS fraud_transactions
FROM reporting.vw_aml_transactions
WHERE "isFraud" = 1
GROUP BY typology
ORDER BY fraud_transactions DESC;
"""

typology_summary = pd.read_sql(typology_query, engine)

typology_summary

,typology,fraud_transactions
0,layering,1005
1,structuring,348
2,integration,58


In [43]:
# Transaction Type Summary

transaction_type_query = """
SELECT
    transaction_type,
    COUNT(*) AS fraud_transactions
FROM reporting.vw_aml_transactions
WHERE "isFraud" = 1
GROUP BY transaction_type
ORDER BY fraud_transactions DESC;
"""

transaction_type_summary = pd.read_sql(transaction_type_query, engine)

transaction_type_summary

,transaction_type,fraud_transactions
0,TRANSFER,1353
1,PAYMENT,58


In [44]:
# Fraud Category Summary

category_query = """
SELECT
    category,
    COUNT(*) AS fraud_transactions
FROM reporting.vw_aml_transactions
WHERE "isFraud" = 1
GROUP BY category
ORDER BY fraud_transactions DESC;
"""

category_summary = pd.read_sql(category_query, engine)

category_summary

,category,fraud_transactions
0,Other,1182
1,Shell Company,117
2,Recreation,94
3,Property Investment,16
4,Cryptocurrency,2


# ========================
# Build AI Prompts
# ========================

In [45]:
# Convert DataFrame into readable bullet text

def dataframe_to_text(df, name_column, value_column):

    lines = []

    for _, row in df.iterrows():

        lines.append(
            f"- {row[name_column]} : {row[value_column]}"
        )

    return "\n".join(lines)

In [46]:
# Convert every Dataframe

merchant_text = dataframe_to_text(
    merchant_summary,
    "merchant_category",
    "transaction_count"
)

typology_text = dataframe_to_text(
    typology_summary,
    "typology",
    "fraud_transactions"
)

transaction_type_text = dataframe_to_text(
    transaction_type_summary,
    "transaction_type",
    "fraud_transactions"
)

category_text = dataframe_to_text(
    category_summary,
    "category",
    "fraud_transactions"
)

In [47]:
# Verify

print(merchant_text)

- electronics : 200398
- fashion : 200117
- gaming : 199935
- groceries : 199786
- travel : 199755


In [48]:
print(typology_text)

- layering : 1005
- structuring : 348
- integration : 58


# Generative AI - Google Gemini AI Connection

In [49]:
# Install below cmd in Terminal
# pip install google-generativeai 
# then install below cmd
# python -m pip install google-generativeai
# Restart Kernel 
# Check connected to this project's venv 

In [ ]:
from google import genai

from config import GEMINI_API_KEY

client = genai.Client(api_key=GEMINI_API_KEY)

print("Gemini Connected Successfully!")

✅ Gemini Connected Successfully!


In [66]:
# Create Banking AI Prompt

banking_prompt = f"""
You are a Senior Banking Business Analyst.

Analyze the following banking transaction data.

Peak Banking Month:
{peak_month.to_string(index=False)}

Merchant Category Summary:
{category_summary.to_string(index=False)}

Requirements:

Only draw conclusions directly supported by the provided data.
Do NOT assume customer behaviour, loyalty, satisfaction or market conditions.

Generate a professional executive summary in the EXACT format below.

Executive Banking Summary

Business Insights
• Insight 1
• Insight 2
• Insight 3
• Insight 4
• Insight 5

Key Trends
• Trend 1
• Trend 2

Recommendation
• One business recommendation

Rules:
- Do NOT use Markdown.
- Do NOT use ** or ###.
- Keep the summary under 200 words.
- Use simple business language suitable for executives.
"""

In [67]:
# Generate Banking AI Summary

banking_response = client.models.generate_content(
    model="gemini-3.1-flash-lite",
    contents=banking_prompt
)

banking_summary = banking_response.text

print(banking_summary)

Executive Banking Summary

Business Insights
• May recorded the highest transaction volume with 85,324 transactions.
• The Other category accounts for the majority of fraud, totaling 1,182 incidents.
• Shell Company transactions represent the second-highest source of fraud at 117 cases.
• Recreation and Property Investment categories show moderate fraud levels of 94 and 16 incidents respectively.
• Cryptocurrency displays the lowest fraud frequency with only 2 reported transactions.

Key Trends
• Fraud is heavily concentrated within the Other merchant category compared to specialized sectors.
• High-risk sectors such as Shell Companies demonstrate a disproportionate impact on fraud metrics relative to investment and digital asset categories.

Recommendation
• Initiate a detailed audit of the Other merchant category to identify and reclassify high-risk transactions currently missing specific categorization.


In [68]:
# Save Banking AI Summary to PostgreSQL

from sqlalchemy import text
from datetime import datetime

with engine.begin() as conn:

    conn.execute(
        text("""
            UPDATE reporting.ai_dashboard_summary
            SET
                summary_text = :summary_text,
                generated_on = :generated_on
            WHERE summary_type = 'Banking'
        """),
        {
            "summary_text": banking_summary,
            "generated_on": datetime.now()
        }
    )

print("Banking AI Summary saved.")

Banking AI Summary saved.


In [69]:
aml_prompt = f"""
You are a Senior Anti-Money Laundering Risk Analyst.

Analyze the following AML and Fraud data.

Fraud KPIs:
{fraud_kpis.to_string(index=False)}

Fraud Typology Summary:
{typology_text}

Fraud Transaction Type Summary:
{transaction_type_text}

Fraud Category Summary:
{category_text}

Requirements:

Only draw conclusions directly supported by the provided data.
Do NOT assume criminal intent beyond the supplied metrics.

Generate the summary in EXACTLY this format.

Executive AML Summary

Business Insights
• Insight 1
• Insight 2
• Insight 3
• Insight 4
• Insight 5

Key Risks
• Risk 1
• Risk 2
• Risk 3

Recommendation
• One AML recommendation

Rules:
- Do NOT use Markdown.
- Do NOT use ** or ###.
- Maximum 200 words.
- Professional executive language.
"""

In [70]:
# Generate AML AI Summary

aml_response = client.models.generate_content(
    model="gemini-3.1-flash-lite",
    contents=aml_prompt
)

aml_summary = aml_response.text

print(aml_summary)

Executive AML Summary

Business Insights
• The data confirms a direct correlation between total fraud events and identified money laundering cases, indicating a high detection rate across the analyzed volume.
• Layering activity represents the primary typology, accounting for approximately 71 percent of total documented cases.
• Transaction analysis reveals a significant concentration in bank transfers, which constitute 96 percent of all flagged fraud events.
• The Shell Company category represents the second largest identified risk segment, highlighting potential vulnerabilities in corporate onboarding.
• The majority of incidents fall under the Other category, suggesting a need for more granular categorization to improve behavioral modeling.

Key Risks
• High frequency of layering suggests complex, multi-stage transaction patterns designed to obfuscate audit trails.
• Over-reliance on electronic transfer channels increases exposure to rapid fund dissipation.
• Potential exploitation 

In [71]:
# Save AML AI Summary to PostgreSQL

with engine.begin() as conn:

    conn.execute(
        text("""
            UPDATE reporting.ai_dashboard_summary
            SET
                summary_text = :summary_text,
                generated_on = :generated_on
            WHERE summary_type = 'AML'
        """),
        {
            "summary_text": aml_summary,
            "generated_on": datetime.now()
        }
    )

print("AML AI Summary saved.")

AML AI Summary saved.


#Verify in PostgreSQL

#SELECT * FROM reporting.ai_dashboard_summary;